In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers


In [ ]:
T_MAX = 100
NUM_CLASSES = 100
D_FEATURE = 126


def build_sign_model():
    model = models.Sequential([
        # 1) Masking to ignore zero‐padded frames
        layers.Masking(mask_value=0.0, input_shape=(T_MAX, D_FEATURE)),

        # 2) First Bi‐LSTM layer (returns full sequence)
        layers.Bidirectional(
            layers.LSTM(
                units=256,
                return_sequences=True,
                dropout=0.3,
                recurrent_dropout=0.3,
                kernel_regularizer=regularizers.l2(1e-4)
            )
        ),

        # 3) Second Bi‐LSTM layer (returns last hidden state only)
        layers.Bidirectional(
            layers.LSTM(
                units=256,
                return_sequences=False,
                dropout=0.3,
                recurrent_dropout=0.3,
                kernel_regularizer=regularizers.l2(1e-4)
            )
        ),

        # 4) Fully‐connected classifier head
        layers.Dense(
            units=256,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        ),
        layers.Dropout(0.5),
        layers.BatchNormalization(),

        # 5) Final softmax
        layers.Dense(units=NUM_CLASSES, activation="softmax")
    ])
    return model

model = build_sign_model()

# Compile with Adam + gradient clipping
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x=X_train,         # shape (N_train, 100, 126)
    y=y_train,         # shape (N_train,)
    batch_size=32,
    epochs=20,
    validation_data=(X_test, y_test)
)